In [ ]:
import kagglehub
path = kagglehub.dataset_download("davidshinn/github-issues")

100%|██████████| 980M/980M [00:10<00:00, 95.8MB/s]

Extracting files...


In [ ]:
import os
print(os.listdir(path))

['github_issues.csv']


In [ ]:
import pandas as pd
import os

csv_file_path = os.path.join(path, 'github_issues.csv')
df = pd.read_csv(csv_file_path, nrows=1000)

print(f"\n✅ Loaded {len(df)} issues")
print(f"📊 Columns: {list(df.columns)}")

# Show first few rows
print("\n📋 Sample data:")
print(df.head(3))

# Check data quality
print("\n🔍 Data quality check:")
print(f"  - Total issues: {len(df)}")
print(f"  - Missing titles: {df['issue_title'].isna().sum()}")
print(f"  - Missing bodies: {df['body'].isna().sum()}")

# Show one full issue
print("\n📝 Example issue:")
print(f"Title: {df.iloc[0]['issue_title']}")
print(f"Body: {df.iloc[0]['body'][:200]}...")


✅ Loaded 1000 issues
📊 Columns: ['issue_url', 'issue_title', 'body']

📋 Sample data:
                                           issue_url  \
0  "https://github.com/zhangyuanwei/node-images/i...   
1     "https://github.com/Microsoft/pxt/issues/2543"   
2  "https://github.com/MatisiekPL/Czekolada/issue...   

                                         issue_title  \
0  can't load the addon. issue to: https://github...   
1  hcl accessibility a11yblocking a11ymas mas4.2....   
2  issue 1265: issue 1264: issue 1261: issue 1260...   

                                                body  
0  can't load the addon. issue to: https://github...  
1  user experience: user who depends on screen re...  
2  ┆attachments: <a href= https:& x2f;& x2f;githu...  

🔍 Data quality check:
  - Total issues: 1000
  - Missing titles: 0
  - Missing bodies: 0

📝 Example issue:
Title: can't load the addon. issue to: https://github.com/zhangyuanwei/node-images/issues error: /lib64/libc.so.6: version glibc_2.14' n

In [ ]:

import re
print("📥 Loading GitHub Issues dataset...")
df = pd.read_csv(csv_file_path)
print(f"✅ Loaded {len(df)} issues")
SAMPLE_SIZE = 10000
RANDOM_STATE = 42

df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)



print("🔗 Merging title and body...")
df['issue_text'] = df['issue_title'].fillna('') + "\n\n" + df['body'].fillna('')

# ===== 2. BASIC CLEANING (Keep errors/logs as signal) =====
print("🧹 Cleaning text...")

def clean_text(text):
    """Clean GitHub issue text - minimal cleaning to preserve signal"""

    # Remove HTML tags (common in GitHub issues)
    text = re.sub(r'<.*?>', ' ', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove URLs (too specific, not useful for clustering)
    text = re.sub(r'http\S+', ' URL ', text)

    # Keep only ASCII (removes weird unicode)
    text = text.encode('ascii', 'ignore').decode('ascii')

    return text.strip()

df['issue_text'] = df['issue_text'].apply(clean_text)

# ===== 3. REMOVE TOO-SHORT ISSUES =====
df = df[df['issue_text'].str.len() > 30]
print(f"✅ After filtering short issues: {len(df)} valid issues")

# ===== 4. ADD SIMULATED METADATA (TRANSPARENT) =====
print("🏷️ Adding simulated metadata...")

# A. Extract repo name from URL
df['repo'] = df['issue_url'].str.extract(r'github\.com/([^/]+/[^/]+)/')

# B. Simulate timestamps (evenly spaced over 6 months)
df['created_at'] = pd.date_range(
    start='2024-01-01',
    periods=len(df),
    freq='2H'  # One issue every 2 hours
)

# C. Simulate repo tiers based on frequency
repo_counts = df['repo'].value_counts()

def assign_tier(repo):
    count = repo_counts.get(repo, 0)
    if count > 50:
        return 'enterprise'
    elif count > 20:
        return 'pro'
    else:
        return 'free'

df['repo_tier'] = df['repo'].apply(assign_tier)

# D. Assign user_weight based on tier
tier_weights = {'enterprise': 3, 'pro': 2, 'free': 1}
df['user_weight'] = df['repo_tier'].map(tier_weights)

# E. Extract severity keywords
SEVERE_KEYWORDS = [
    'crash', 'segfault', 'data loss', 'payment', 'security',
    'panic', 'fatal', 'error', 'broken', 'not working',
    'timeout', 'failed', 'exception'
]

def keyword_severity(text):
    """Count severe keywords in text"""
    text_lower = text.lower()
    return sum(1 for kw in SEVERE_KEYWORDS if kw in text_lower)

df['keyword_severity'] = df['issue_text'].apply(keyword_severity)

# ===== 5. SAVE CLEANED DATA =====
os.makedirs('data', exist_ok=True)
df.to_csv('data/cleaned_issues.csv', index=False)
print("✅ Saved to data/cleaned_issues.csv")

# ===== 6. SHOW STATISTICS =====
print("\n📊 Dataset Statistics:")
print(f"  - Total issues: {len(df)}")
print(f"  - Average text length: {df['issue_text'].str.len().mean():.0f} chars")
print(f"  - Unique repos: {df['repo'].nunique()}")
print(f"  - Tier distribution:")
print(df['repo_tier'].value_counts())
print(f"\n  - Top 5 repos:")
print(df['repo'].value_counts().head())

# ===== 7. SHOW SAMPLE CLEANED ISSUES =====
print("\n📝 Sample cleaned issues:")
for i in range(3):
    print(f"\n{'='*70}")
    print(f"Issue {i+1}:")
    print(f"  Title: {df.iloc[i]['issue_title'][:60]}...")
    print(f"  Repo: {df.iloc[i]['repo']}")
    print(f"  Tier: {df.iloc[i]['repo_tier']} (weight: {df.iloc[i]['user_weight']})")
    print(f"  Severity keywords: {df.iloc[i]['keyword_severity']}")
    print(f"  Text preview: {df.iloc[i]['issue_text'][:150]}...")

print(f"\n{'='*70}")
print("✅ WEEK 1 DATA PREPARATION COMPLETE")
print("\n💡 Note: Timestamps and repo tiers are simulated for demonstration.")
print("   In production, these would come from your SaaS platform's metadata.")

📥 Loading GitHub Issues dataset...
✅ Loaded 5332153 issues
🔗 Merging title and body...
🧹 Cleaning text...
✅ After filtering short issues: 9961 valid issues
🏷️ Adding simulated metadata...


/tmp/ipython-input-3130224000.py:48: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['created_at'] = pd.date_range(


✅ Saved to data/cleaned_issues.csv

📊 Dataset Statistics:
  - Total issues: 9961
  - Average text length: 413 chars
  - Unique repos: 8212
  - Tier distribution:
repo_tier
free          9637
enterprise     166
pro            158
Name: count, dtype: int64

  - Top 5 repos:
repo
koorellasuresh/UKRegionTest    166
lstjsuperman/fabric             49
shuhongwu/hockeyapp             49
Microsoft/vscode                35
chrmarti/testissues             25
Name: count, dtype: int64

📝 Sample cleaned issues:

Issue 1:
  Title: manually entered dates issues...
  Repo: atais/angular-eonasdan-datetimepicker
  Tier: free (weight: 1)
  Severity keywords: 1
  Text preview: manually entered dates issues i use format 'yyyy-mm-dd' option and bind ng-model= formdata.date to my field. now if i enter '2015-03-05aaaaa', the dat...

Issue 2:
  Title: highlight segment on map when editing speed...
  Repo: conveyal/analysis-ui
  Tier: free (weight: 1)
  Severity keywords: 0
  Text preview: highlight segment on

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load cleaned data
df = pd.read_csv('data/cleaned_issues.csv')
print(f"✅ Loaded {len(df)} issues")

# Load embedding model
print("📥 Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded")

# Generate embeddings in batches (to avoid memory issues)
print("🔄 Generating embeddings...")

batch_size = 100  # Process 100 issues at a time
all_embeddings = []

for i in tqdm(range(0, len(df), batch_size)):
    batch = df['issue_text'].iloc[i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.append(batch_embeddings)

# Combine all batches
embeddings = np.vstack(all_embeddings)

# Save embeddings
np.save('data/embeddings.npy', embeddings)
print(f"✅ Saved embeddings: shape {embeddings.shape}")
print(f"   {len(df)} issues × 384 dimensions")

# Memory usage
size_mb = embeddings.nbytes / (1024 * 1024)
print(f"   File size: {size_mb:.1f} MB")

# Quick sanity check
print("\n🔍 Sanity check:")
print(f"  - First embedding (sample): {embeddings[0][:5]}")
print(f"  - All values finite: {np.isfinite(embeddings).all()}")
print(f"  - Range: [{embeddings.min():.3f}, {embeddings.max():.3f}]")

✅ Loaded 9961 issues
📥 Loading embedding model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded
🔄 Generating embeddings...


100%|██████████| 100/100 [11:01<00:00,  6.61s/it]

✅ Saved embeddings: shape (9961, 384)
   9961 issues × 384 dimensions
   File size: 14.6 MB

🔍 Sanity check:
  - First embedding (sample): [-0.04180107  0.01060041  0.0458297   0.08161619 -0.0523266 ]
  - All values finite: True
  - Range: [-0.268, 0.287]


In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load model and data
print("📥 Loading data...")
model = SentenceTransformer('all-MiniLM-L6-v2')
df = pd.read_csv('data/cleaned_issues.csv')
embeddings = np.load('data/embeddings.npy')
print(f"✅ Ready to search {len(df)} issues")

def search_issues(query, top_k=5):
    """Find most similar issues to query"""

    # Convert query to embedding
    query_embedding = model.encode([query])

    # Calculate similarity scores
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Get top matches
    top_indices = similarities.argsort()[-top_k:][::-1]

    print(f"\n{'='*70}")
    print(f"🔍 SEARCH: '{query}'")
    print(f"{'='*70}")

    for i, idx in enumerate(top_indices, 1):
        issue = df.iloc[idx]
        similarity = similarities[idx]

        print(f"\n[{i}] Similarity: {similarity:.3f}")
        print(f"    Title: {issue['issue_title']}") # Corrected from 'title' to 'issue_title'

        if 'repo' in df.columns:
            print(f"    Repo: {issue['repo']}")

        # Show first 200 chars of body
        body_preview = issue['issue_text'][:200].replace('\n', ' ')
        print(f"    Text: {body_preview}...")

    print(f"\n{'='*70}\n")

# Test searches with common issue types
print("\n🎯 Testing semantic search on real GitHub issues:\n")

search_issues("app crashes when opening", top_k=3)
search_issues("login authentication error", top_k=3)
search_issues("database connection timeout", top_k=3)
search_issues("UI button not responding", top_k=3)

# Interactive search (optional)
print("\n💡 Try your own search:")
while True:
    user_query = input("\nEnter search query (or 'quit' to exit): ")
    if user_query.lower() == 'quit':
        break
    search_issues(user_query, top_k=3)


📥 Loading data...
✅ Ready to search 9961 issues

🎯 Testing semantic search on real GitHub issues:


🔍 SEARCH: 'app crashes when opening'

[1] Similarity: 0.713
    Title: crashes on first start
    Repo: samvrlewis/ozbargain-notify
    Text: crashes on first start after installing/updating, the app crashes. closing and then relaunching seems to fix. may only be a problem on android n....

[2] Similarity: 0.652
    Title: app crashing !! in xcode 8.3 beta 3
    Repo: eni9889/ppsideloader
    Text: app crashing !! in xcode 8.3 beta 3 hi , i have tried whatsapp++ , youtube++ , twitter++ every app is crashing after installing , when i open any one is close quickly how can i fix this issue ?...

[3] Similarity: 0.615
    Title: applicationpackagemanager.java line 137
    Repo: lstjsuperman/fabric
    Text: applicationpackagemanager.java line 137 in android.app.applicationpackagemanager.getpackageinfo number of crashes: 1 impacted devices: 1 there's a lot more information about this crash on

In [ ]:
import pandas as pd
import numpy as np
import os

print("🔍 WEEK 1 VALIDATION CHECKLIST\n")

# Check all required files exist
files_to_check = {
    'data/cleaned_issues.csv': 'Cleaned dataset',
    'data/embeddings.npy': 'Issue embeddings',
}

all_good = True

for file, description in files_to_check.items():
    exists = os.path.exists(file)
    size_mb = os.path.getsize(file) / (1024*1024) if exists else 0

    status = "✅" if exists else "❌"
    print(f"{status} {description}: {file}")
    if exists:
        print(f"   Size: {size_mb:.1f} MB")

    all_good = all_good and exists

# Load and validate data
if all_good:
    df = pd.read_csv('data/cleaned_issues.csv')
    embeddings = np.load('data/embeddings.npy')

    print(f"\n📊 Data Validation:")
    print(f"  ✅ {len(df)} issues loaded")
    print(f"  ✅ {embeddings.shape[0]} embeddings generated")
    print(f"  ✅ Match: {len(df) == embeddings.shape[0]}")

    print(f"\n🎯 Ready for Week 2: Clustering & Root Cause Detection")
else:
    print(f"\n❌ Some files missing. Run the scripts in order:")
    print(f"   1. week1_download_data.py")
    print(f"   2. week1_load_data.py")
    print(f"   3. week1_embeddings.py")

🔍 WEEK 1 VALIDATION CHECKLIST

✅ Cleaned dataset: data/cleaned_issues.csv
   Size: 9.5 MB
✅ Issue embeddings: data/embeddings.npy
   Size: 14.6 MB

📊 Data Validation:
  ✅ 9961 issues loaded
  ✅ 9961 embeddings generated
  ✅ Match: True

🎯 Ready for Week 2: Clustering & Root Cause Detection


In [ ]:
import pandas as pd
import re

print("🔍 MULTIMODAL FEATURE AUDIT\n")

# Load your cleaned data
df = pd.read_csv('data/cleaned_issues.csv')

print(f"Dataset: {len(df)} issues\n")
print("="*70)

# ===== CHECK 1: What columns do we have? =====
print("\n📋 AVAILABLE COLUMNS:")
print(df.columns.tolist())

# ===== CHECK 2: Image/Screenshot presence =====
print("\n\n🖼️ IMAGE ANALYSIS:")

# Common image patterns in GitHub markdown
image_patterns = [
    r'!\[.*?\]\(.*?\)',           # ![alt](url) markdown
    r'<img[^>]+>',                # <img> HTML tags
    r'https?://[^\s]+\.(?:png|jpg|jpeg|gif|svg)',  # Direct image URLs
]

df['has_image'] = df['body'].fillna('').str.contains('|'.join(image_patterns), regex=True)

print(f"  - Issues with images: {df['has_image'].sum()} ({df['has_image'].sum()/len(df)*100:.1f}%)")

if df['has_image'].sum() > 0:
    print("\n  Sample issues with images:")
    for i, row in df[df['has_image']].head(3).iterrows():
        print(f"    • {row['issue_title'][:60]}...")

# ===== CHECK 3: Code blocks =====
print("\n\n💻 CODE BLOCK ANALYSIS:")

code_patterns = [
    r'```',                       # Markdown code blocks
    r'`[^`]+`',                   # Inline code
    r'<code>',                    # HTML code tags
]

df['has_code'] = df['body'].fillna('').str.contains('|'.join(code_patterns), regex=True)

print(f"  - Issues with code: {df['has_code'].sum()} ({df['has_code'].sum()/len(df)*100:.1f}%)")

# ===== CHECK 4: URLs/Links =====
print("\n\n🔗 URL ANALYSIS:")

df['url_count'] = df['body'].fillna('').str.count(r'https?://')

print(f"  - Issues with URLs: {(df['url_count'] > 0).sum()} ({(df['url_count'] > 0).sum()/len(df)*100:.1f}%)")
print(f"  - Average URLs per issue: {df['url_count'].mean():.2f}")
print(f"  - Max URLs in one issue: {df['url_count'].max()}")

# ===== CHECK 5: Stack traces/Error logs =====
print("\n\n⚠️ ERROR PATTERN ANALYSIS:")

error_patterns = [
    r'error:',
    r'exception',
    r'traceback',
    r'at line \d+',
    r'stack trace',
]

df['has_error_trace'] = df['issue_text'].str.lower().str.contains('|'.join(error_patterns), regex=True)

print(f"  - Issues with error traces: {df['has_error_trace'].sum()} ({df['has_error_trace'].sum()/len(df)*100:.1f}%)")

# ===== CHECK 6: Issue references (links to other issues) =====
print("\n\n🔗 ISSUE REFERENCE ANALYSIS:")

issue_ref_pattern = r'#\d+|issue \d+|https://github\.com/[^/]+/[^/]+/issues/\d+'
df['issue_refs'] = df['body'].fillna('').str.findall(issue_ref_pattern)
df['issue_ref_count'] = df['issue_refs'].apply(len)

print(f"  - Issues referencing other issues: {(df['issue_ref_count'] > 0).sum()} ({(df['issue_ref_count'] > 0).sum()/len(df)*100:.1f}%)")
print(f"  - Average references per issue: {df['issue_ref_count'].mean():.2f}")

if df['issue_ref_count'].max() > 0:
    print(f"\n  Top issue with most references ({df['issue_ref_count'].max()} refs):")
    top_ref = df.loc[df['issue_ref_count'].idxmax()]
    print(f"    Title: {top_ref['issue_title'][:60]}...")
    print(f"    Refs: {top_ref['issue_refs'][:5]}")

# ===== CHECK 7: Text characteristics =====
print("\n\n📝 TEXT CHARACTERISTICS:")

df['text_length'] = df['issue_text'].str.len()
df['word_count'] = df['issue_text'].str.split().str.len()
df['has_question'] = df['issue_text'].str.contains(r'\?')

print(f"  - Avg text length: {df['text_length'].mean():.0f} chars")
print(f"  - Avg word count: {df['word_count'].mean():.0f} words")
print(f"  - Issues with questions: {df['has_question'].sum()} ({df['has_question'].sum()/len(df)*100:.1f}%)")

# ===== CHECK 8: Existing labels (if any) =====
print("\n\n🏷️ LABEL ANALYSIS:")

if 'labels' in df.columns:
    df['has_labels'] = df['labels'].notna() & (df['labels'] != '')
    print(f"  - Issues with labels: {df['has_labels'].sum()} ({df['has_labels'].sum()/len(df)*100:.1f}%)")

    if df['has_labels'].sum() > 0:
        # Try to parse labels
        all_labels = []
        for labels in df[df['has_labels']]['labels']:
            if isinstance(labels, str):
                all_labels.extend([l.strip() for l in labels.split(',')])

        if all_labels:
            from collections import Counter
            label_counts = Counter(all_labels)
            print(f"\n  Top 5 labels:")
            for label, count in label_counts.most_common(5):
                print(f"    • {label}: {count}")
else:
    print(f"  - No 'labels' column in dataset")

# ===== SUMMARY: Available Multimodal Features =====
print("\n\n" + "="*70)
print("📊 MULTIMODAL FEATURE SUMMARY")
print("="*70)

features = {
    'Text embeddings': '✅ Already have (9916 × 384)',
    'Image presence': f"{'✅' if df['has_image'].sum() > 100 else '⚠️'} {df['has_image'].sum()} issues",
    'Code blocks': f"✅ {df['has_code'].sum()} issues",
    'Error traces': f"✅ {df['has_error_trace'].sum()} issues",
    'Issue references': f"{'✅' if df['issue_ref_count'].sum() > 100 else '⚠️'} {(df['issue_ref_count'] > 0).sum()} issues",
    'URL count': f"✅ Numeric feature (0-{df['url_count'].max()})",
    'Text length': f"✅ Numeric feature ({df['text_length'].min()}-{df['text_length'].max()})",
    'Severity keywords': '✅ Already have',
    'User weight': '✅ Already have',
    'Repo tier': '✅ Already have',
}

for feature, status in features.items():
    print(f"  {status:30s} {feature}")

# ===== Save augmented data =====
print("\n\n💾 Saving augmented data...")

df.to_csv('data/cleaned_issues_with_features.csv', index=False)

print("✅ Saved to: data/cleaned_issues_with_features.csv")
print("\n🎯 Ready for multimodal clustering in Week 2!")

🔍 MULTIMODAL FEATURE AUDIT

Dataset: 9961 issues


📋 AVAILABLE COLUMNS:
['issue_url', 'issue_title', 'body', 'issue_text', 'repo', 'created_at', 'repo_tier', 'user_weight', 'keyword_severity']


🖼️ IMAGE ANALYSIS:
  - Issues with images: 910 (9.1%)

  Sample issues with images:
    • build errors on latest version...
    • new import page bug - page titles...
    • feedback has been mispelled as 'feeback'...


💻 CODE BLOCK ANALYSIS:
  - Issues with code: 14 (0.1%)


🔗 URL ANALYSIS:
  - Issues with URLs: 3379 (33.9%)
  - Average URLs per issue: 0.60
  - Max URLs in one issue: 34


⚠️ ERROR PATTERN ANALYSIS:
  - Issues with error traces: 896 (9.0%)


🔗 ISSUE REFERENCE ANALYSIS:
  - Issues referencing other issues: 228 (2.3%)
  - Average references per issue: 0.03

  Top issue with most references (10 refs):
    Title: support more means of payin and payout...
    Refs: ['https://github.com/gratipay/gratipay.com/issues/3870', 'https://github.com/gratipay/gratipay.com/issues/3870', 'https: